In [10]:
from datasets import load_dataset

ds = load_dataset("westenfelder/NL2SH-ALFA", "train")

def create_conversation(sample):
  return {
      "messages": [
          {"role": "system", "content": "You are a helpful assistant that translates natural language to bash commands."},
          {"role": "user", "content": "Generate single Bash command: " + sample["nl"]},
          {"role": "assistant", "content": f'```bash\n{sample["bash"]}\n```'},
      ]
  }

train_dataset = ds['train'].map(create_conversation, remove_columns=ds['train'].features, batched=False)

dataset = train_dataset.train_test_split(test_size=0.2, shuffle=True)


Map: 100%|██████████| 40639/40639 [00:00<00:00, 43513.13 examples/s]


In [11]:
print(dataset["train"][0]["messages"])

[{'content': 'You are a helpful assistant that translates natural language to bash commands.', 'role': 'system'}, {'content': 'Generate single Bash command: Determine the MTU to the destination', 'role': 'user'}, {'content': '```bash\ntraceroute --mtu example.com\n```', 'role': 'assistant'}]


In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

# Load model and tokenizer
model = AutoModelForCausalLM.from_pretrained(
    "google/gemma-3-270m-it",
    dtype="auto",
    device_map="cuda",
    attn_implementation="eager"
)
tokenizer = AutoTokenizer.from_pretrained("google/gemma-3-270m-it")

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

# Hugging Face model id
model_id = "google/gemma-3-270m-it"

# Check if GPU benefits from bfloat16
if torch.cuda.get_device_capability()[0] >= 8:
    torch_dtype = torch.bfloat16
else:
    torch_dtype = torch.float16

# Define model init arguments
model_kwargs = dict(
    attn_implementation="eager", # Use "flash_attention_2" when running on Ampere or newer GPU
    dtype=torch_dtype, # What torch dtype to use, defaults to auto
    device_map="auto", # Let torch decide how to load the model
)

# BitsAndBytesConfig: Enables 4-bit quantization to reduce model size/memory usage
model_kwargs["quantization_config"] = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=model_kwargs['torch_dtype'],
    bnb_4bit_quant_storage=model_kwargs['torch_dtype'],
)


print(f"Device: {model.device}")
print(f"DType: {model.dtype}")

# Load model and tokenizer
model = AutoModelForCausalLM.from_pretrained(model_id, **model_kwargs)
tokenizer = AutoTokenizer.from_pretrained(model_id) # Load the Instruction Tokenizer to use the official Gemma template

Device: cuda:0
DType: torch.bfloat16


In [ ]:
from peft import LoraConfig

peft_config = LoraConfig(
    lora_alpha=16,
    lora_dropout=0.05,
    r=16,
    bias="none",
    target_modules="all-linear",
    task_type="CAUSAL_LM",
    modules_to_save=["lm_head", "embed_tokens"] # make sure to save the lm_head and embed_tokens as you train the special tokens
)

In [ ]:
from trl import SFTConfig

args = SFTConfig(
    output_dir=f"{model_id}-bash-qlora",         # directory to save and repository id
    max_length=512,                         # max sequence length for model and packing of the dataset
    packing=True,                           # Groups multiple samples in the dataset into a single sequence
    num_train_epochs=3,                     # number of training epochs
    per_device_train_batch_size=1,          # batch size per device during training
    gradient_accumulation_steps=4,          # number of steps before performing a backward/update pass
    gradient_checkpointing=True,            # use gradient checkpointing to save memory
    optim="adamw_torch_fused",              # use fused adamw optimizer
    logging_steps=10,                       # log every 10 steps
    save_strategy="epoch",                  # save checkpoint every epoch
    learning_rate=2e-4,                     # learning rate, based on QLoRA paper
    fp16=True if torch_dtype == torch.float16 else False,   # use float16 precision
    bf16=True if torch_dtype == torch.bfloat16 else False,   # use bfloat16 precision
    max_grad_norm=0.3,                      # max gradient norm based on QLoRA paper
    warmup_ratio=0.03,                      # warmup ratio based on QLoRA paper
    lr_scheduler_type="constant",           # use constant learning rate scheduler
    push_to_hub=True,                       # push model to hub
    report_to="tensorboard",                # report metrics to tensorboard
    dataset_kwargs={
        "add_special_tokens": False, # We template with special tokens
        "append_concat_token": True, # Add EOS token as separator token between examples
    }
)

Device set to use cuda


Question:
Generate single Bash command: Find all files in the current directory, excluding those beginning with "#", list their details in long format, and sort them in reverse order by their fourth field.

Original Answer:
grep -vE "^#" <(find $(echo * -maxdepth 0) -type f) | xargs ls -l | sort -nt,2 -k4 -r

Generated Answer (base model):
```bash
find. -maxdepth 1 -type f -not -not -not -not -not -not -not -not -not -not -not -not -not -not -not -not -not -not -not -not -not -not -not -not -not -not -not -not -not -not -not -not -not -not -not -not -not -not -not -not -not -not -not -not -not -not -not -not -not -not -not -not -not -not -not -not -not -not -not -not -not -not -not -not -not -not -not -not -not -not -not -not -not -not -not -not -not -not -not -not -not -not -not -not -not -not -not -not -not -not -not -not -not -not -not -not -not -not -not -not -not -not -not -not -not -not -not -not -not -not -not -not -not -not -not -not -not -not -not -not -not -


In [ ]:
from trl import SFTTrainer

# Create Trainer object
trainer = SFTTrainer(
    model=model,
    args=args,
    train_dataset=dataset["train"],
    peft_config=peft_config,
    processing_class=tokenizer
)

[{'role': 'system', 'content': 'You are a helpful assistant that translates natural language to bash commands.'}, {'role': 'user', 'content': 'Generate single Bash command: Find all files in the current directory, excluding those beginning with "#", list their details in long format, and sort them in reverse order by their fourth field.'}]
[{'role': 'system', 'content': 'You are a helpful assistant that translates natural language to bash commands.'}, {'role': 'user', 'content': 'Generate single Bash command: Find all files in the current directory, excluding those beginning with "#", list their details in long format, and sort them in reverse order by their fourth field.'}, {'role': 'assistant', 'content': '```bash\nfind. -type f -not -not -not -not -not -not -not -not -not -not -not -not -not -not -not -not -not -not -not -not -not -not -not -not -not -not -not -not -not -not -not -not -not -not -not -not -not -not -not -not -not -not -not -not -not -not -not -not -not -not -not -not

In [ ]:
# Start training, the model will be automatically saved to the Hub and the output directory
trainer.train()

# Save the final model again to the Hugging Face Hub
trainer.save_model()

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': 2, 'pad_token_id': 0}.


Epoch,Training Loss,Validation Loss,Entropy,Num Tokens,Mean Token Accuracy
1,0.992600,0.938504,0.918691,2188280.000000,0.803903
2,0.672600,0.898868,0.752329,4376560.000000,0.813216
3,0.912200,0.913020,0.661497,6564840.000000,0.816186


In [ ]:
# free the memory again
del model
del trainer
torch.cuda.empty_cache()

In [ ]:
from peft import PeftModel

# Load Model base model
model = AutoModelForCausalLM.from_pretrained(model_id, low_cpu_mem_usage=True)

# Merge LoRA and base model and save
peft_model = PeftModel.from_pretrained(model, args.output_dir)
merged_model = peft_model.merge_and_unload()
merged_model.save_pretrained("merged_model", safe_serialization=True, max_shard_size="2GB")

processor = AutoTokenizer.from_pretrained(args.output_dir)
processor.save_pretrained("merged_model")

In [ ]:
import torch
from transformers import pipeline

model_id = f"{model_id}-bash-qlora"

# Load Model with PEFT adapter
model = AutoModelForCausalLM.from_pretrained(
  model_id,
  device_map="auto",
  torch_dtype=torch_dtype,
  attn_implementation="eager",
)
tokenizer = AutoTokenizer.from_pretrained(model_id)

In [ ]:
from random import randint
import re

# Load the model and tokenizer into the pipeline
pipe = pipeline("text-generation", model=model, tokenizer=tokenizer)

# Load a random sample from the test dataset
rand_idx = randint(0, len(dataset["test"])-1)
test_sample = dataset["test"][rand_idx]

# Convert as test example into a prompt with the Gemma template
stop_token_ids = [tokenizer.eos_token_id, tokenizer.convert_tokens_to_ids("<end_of_turn>")]
prompt = pipe.tokenizer.apply_chat_template(test_sample["messages"][:2], tokenize=False, add_generation_prompt=True)

# Generate our SQL query.
outputs = pipe(prompt, max_new_tokens=256, do_sample=False, temperature=0.1, top_k=50, top_p=0.1, eos_token_id=stop_token_ids, disable_compile=True)

# Extract the user query and original answer
print(f"Context:\n", re.search(r'<SCHEMA>\n(.*?)\n</SCHEMA>', test_sample['messages'][0]['content'], re.DOTALL).group(1).strip())
print(f"Query:\n", re.search(r'<USER_QUERY>\n(.*?)\n</USER_QUERY>', test_sample['messages'][0]['content'], re.DOTALL).group(1).strip())
print(f"Original Answer:\n{test_sample['messages'][1]['content']}")
print(f"Generated Answer:\n{outputs[0]['generated_text'][len(prompt):].strip()}")